In [1]:
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torch

class SpatialClusterHead(nn.Module):
    def __init__(self, feature_dim, num_clusters):
        super().__init__()
        self.spatial_mlp = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, num_clusters)
        )
        self.feature_mlp = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_clusters)
        )
        
        # THE FIX: Equalizers
        # This prevents the feature_mlp from bullying the spatial_mlp
        self.spatial_norm = nn.LayerNorm(num_clusters)
        self.feature_norm = nn.LayerNorm(num_clusters)
        
    def forward(self, X_combined, spatial_weight=3.0):
        spatial_logits = self.spatial_mlp(X_combined[:, :2])
        feature_logits = self.feature_mlp(X_combined)
        
        # Normalize both to unit variance before combining
        s = spatial_logits / (spatial_logits.std() + 1e-8)
        f = feature_logits / (feature_logits.std() + 1e-8)
        
        
        return spatial_weight * s + f

class FirstTerm(nn.Module):
    def __init__(self, num_cell_types, num_of_clusters , embedding_dim=8):
        super().__init__()
        self.cell_embedding = nn.Embedding(num_cell_types, embedding_dim)
        # Initialize 10.9M weights (The Dials)
        feature_dim = 18 + embedding_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(feature_dim*2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        nn.init.normal_(self.edge_mlp[2].weight, mean=0.0, std=0.001)
        nn.init.constant_(self.edge_mlp[2].bias, 0.01)


        # 3. THE DUAL-PATH CLUSTER HEAD
        # Replaces the standard Sequential MLP
        self.cluster_head = SpatialClusterHead(
            feature_dim=feature_dim, 
            num_clusters=num_of_clusters
        )



    def forward(self, X, X_cell_ids, num_nodes, p_indices, A_skip_csr, current_k, tau=1.0):
        
        X_cell_ids = X_cell_ids.squeeze()
        cell_features = self.cell_embedding(X_cell_ids)  # Shape: [num_nodes, embedding_dim]
        X_combined = torch.cat([X, cell_features], dim=1)  # Shape: [num_nodes, 18 + embedding_dim]

        src_features = X_combined[p_indices[0]]  # Shape: [num_edges, feature_dim]
        dst_features = X_combined[p_indices[1]]  # Shape: [num_edges, feature_dim]
        edge_inputs = torch.cat([src_features, dst_features], dim=1)  # Shape: [num_edges, feature_dim*2]

        dynamic_p_weights = self.edge_mlp(edge_inputs).squeeze(-1)
        safe_weights = F.softplus(dynamic_p_weights) 


        


        # Enforce P >= 0 and build sparse matrix
        P = torch.sparse_coo_tensor(p_indices, safe_weights, 
                                    (num_nodes, num_nodes)).coalesce()
        
        # Reconstruction: XP
        X_hat = torch.sparse.mm(P, X)
        
        # Loss: ||X - XP||
        error = X - X_hat
        loss1 = torch.mean(error**2)   

        # Pass all node features through the head
        logits = self.cluster_head(X_combined) # Shape: [n, k]

        logits = logits[:, :current_k]
        
        #TERM2
        #C matrix with probability distribution across clusters for each node
        C = F.gumbel_softmax(logits, tau=tau, hard=False) # Shape: [n, k]
        # Inside FirstTerm.forward or the training loop
        # tau_smooth = 0.6  # Higher = more spread out/soft
        # C = F.softmax(logits / tau_smooth, dim=-1)
        # C = F.softmax(logits, dim=-1)  # Ensure positivity for SDDMM

        p_vals = P.values()
        
        # 1. Sum across rows (dim=1) to get the total weight leaving each node
        row_sums = torch.sparse.sum(P, dim=1).to_dense()
        
        # 2. Expand row_sums to match the non-zero values 
        # P.indices()[0] contains the row index for every specific edge
        p_vals_norm = p_vals / (row_sums[P.indices()[0]] + 1e-8)
        
        # Rebuild using the exact same sorted indices
        P_norm = torch.sparse_coo_tensor(P.indices(), p_vals_norm, 
                                         (num_nodes, num_nodes)).coalesce()
        
        # 2. Convert to CSR format (Required for the CUDA SDDMM engine)
        P_csr = P_norm.to_sparse_csr()
        
        # 3. SDDMM Magic! 
        # beta=1.0, alpha=-1.0 calculates exactly: (1.0 * P_csr) - (1.0 * C @ C^T)
        # It ONLY calculates this at the 10.9M non-zero locations!
        diff_csr = torch.sparse.sampled_addmm(P_csr, C, C.t(), beta=1.0, alpha=-1.0)
        
        # 4. Square the differences and sum them
        loss2 = torch.sum(diff_csr.values() ** 2)


        #TERM3
        M = torch.matmul(C.t(), torch.sparse.mm(A_skip_csr, C))  # [k, n] @ [n, n] @ [n, k] -> [k, k]
        # 2. Normalize M into a probability distribution (M_tilde)
        M = torch.clamp(M, min=0)
        M_tilde = M / (M.sum() + 1e-8)
        loss3 = -torch.sum(M_tilde * torch.log(M_tilde + 1e-8))

        # 3. Calculate Shannon Entropy: -sum(p * log(p))
        # We only calculate for non-zero entries to avoid log(0)
        # loss3 = torch.sum(M_tilde * torch.log(M_tilde + 1e-8))

        

        alpha_1 = 1.0 
        alpha_2 = 1.0    
        alpha_3 = 1.0    
        loss = alpha_1* loss1 + (alpha_2 * loss2) +(alpha_3 * loss3) 



        return  loss, loss1 , loss2, loss3,  C , X_combined


import math
def get_tau(epoch):
    tau_start = 2.0
    tau_mid = 1.35   # Targets ~25% Confidence for the middle flatline
    tau_end = 0.85   # Targets ~40% Confidence for the final floor

    if epoch < 75:
        # Phase 1: Smooth descent to the middle flatline
        progress = epoch / 75.0
        return tau_mid + 0.5 * (tau_start - tau_mid) * (1 + math.cos(math.pi * progress))
        
    elif epoch < 150:
        # Phase 2: THE MIDDLE FLATLINE (Epoch 75 to 150)
        return tau_mid
        
    elif epoch < 200:
        # Phase 3: Smooth descent to the final floor
        progress = (epoch - 150) / 50.0
        return tau_end + 0.5 * (tau_mid - tau_end) * (1 + math.cos(math.pi * progress))
        
    else:
        # Phase 4: THE FINAL FLATLINE (Epoch 200 to 250+)
        return tau_end
    


import math

def get_true_compactness_loss(C, positions):
    """
    Dynamically balances tight spatial grouping with 100% cluster utilization.
    Guaranteed to stay positive. Perfect utilization = 0.0 penalty.
    """
    # ==========================================
    # FORCE 1: THE VACUUM (Smooth Spatial Penalty)
    # ==========================================
    weights = C.sum(dim=0) + 1e-8  
    centroids = (C.T @ positions) / weights.unsqueeze(1)  
    
    pos_sq = torch.sum(positions**2, dim=1, keepdim=True)
    cent_sq = torch.sum(centroids**2, dim=1)
    dists_sq = pos_sq + cent_sq - 2 * (positions @ centroids.T)
    
    spatial_loss = (C * (dists_sq ** 2)).sum(dim=1).mean()
    
    # ==========================================
    # FORCE 2: THE WATER PRESSURE (Utilization Reward)
    # ==========================================
    avg_probs = C.mean(dim=0)  # [K]
    K = C.size(1)
    
    # The maximum possible entropy is exactly log(K)
    max_entropy = math.log(K)
    
    # This value is naturally negative
    neg_entropy = torch.sum(avg_probs * torch.log(avg_probs + 1e-10))
    
    # By adding them, perfect balance = 0.0. Complete collapse = +max_entropy.
    utilization_loss = max_entropy + neg_entropy
    
    # ==========================================
    # DYNAMIC BALANCE
    # ==========================================
    # Because utilization_loss is now a smaller positive number (0 to ~6.8),
    # you might need to slightly bump gamma if it doesn't utilize all clusters.
    gamma = 5.0 
    
    return spatial_loss + (gamma * utilization_loss)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F




In [3]:
import os , random
processed_dir = "processed_graphs"
# List all processed file names
design_files = [f for f in os.listdir(processed_dir) if f.endswith('.pt')]

grouped_designs = {}
for f in design_files:
    # Splits "aes_run_20260306..." to extract just "aes"
    base_name = f.split('_run_')[0] 
    if base_name not in grouped_designs:
        grouped_designs[base_name] = []
    grouped_designs[base_name].append(f)

# 2. Pick exactly 20 from each category
UNSEEN_DESIGN = "aes" 

train_files = []
test_files = []

for name, files in grouped_designs.items():
    random.shuffle(files)
    if name == UNSEEN_DESIGN:
        # Truly unseen: keep all AES runs for testing
        test_files.extend(files) 
    else:
        # Standard training: Take 20 balanced samples from others
        selected = files[:20]
        train_files.extend(selected)

random.shuffle(train_files)
random.shuffle(test_files)

# 2. RAM CACHING
train_cache = []
test_cache = []

print(f"📦 Pre-loading TRAIN cache ({len(train_files)} designs)...")
for f in train_files[:20]:
    data = torch.load(os.path.join(processed_dir, f), map_location='cpu')
    train_cache.append((f, data))

print(f"📦 Pre-loading TEST cache ({len(test_files)} designs from {UNSEEN_DESIGN})...")
for f in test_files[:5]:
    data = torch.load(os.path.join(processed_dir, f), map_location='cpu')
    test_cache.append((f, data))

print(f"✅ Pre-loading complete! Training on {len(train_cache)} | Testing on {len(test_cache)}")
 

📦 Pre-loading TRAIN cache (60 designs)...


/home/rain/CTS-Task-Aware-Clustering/venv/lib/python3.12/site-packages/torch/_utils.py:361: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  result = torch.sparse_compressed_tensor(


📦 Pre-loading TEST cache (31 designs from aes)...
✅ Pre-loading complete! Training on 20 | Testing on 5


In [4]:
import torch
import os

# 1. Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Load global metadata (needed for model initialization)
metadata = torch.load("global_metadata.pt")
global_max_cell_types = metadata['global_max_cell_types']
global_max_k = metadata['global_max_k']

# 3. Initialize the Phase 1 Model
# (Assuming your FirstTerm and SpatialClusterHead classes are defined above this)
phase1_model = FirstTerm(
    num_cell_types=global_max_cell_types, 
    num_of_clusters=global_max_k
).to(device)

# 4. Load the trained weights
phase1_model.load_state_dict(torch.load("phase1_clustering_ep250_twohop.pt", map_location=device))

# 5. 🔴 CRITICAL: Freeze the model
phase1_model.eval() # Disables dropout and locks batchnorm stats
for param in phase1_model.parameters():
    param.requires_grad = False

print("✅ Phase 1 Model loaded and frozen. Ready to extract C matrices.")

✅ Phase 1 Model loaded and frozen. Ready to extract C matrices.


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TaskPredictor(nn.Module):
    def __init__(self, feat_dim, num_knobs=4, hidden=128):
        super().__init__()
        # Shared context for the tool knobs
        self.knob_proj = nn.Linear(num_knobs, 32)
        
        # 1. SKEW HEAD: Predicts local skew bottleneck
        self.skew_mlp = nn.Sequential(
            nn.Linear(feat_dim + 32, hidden//2),
            nn.ReLU(),
            nn.Linear(hidden//2, 1)
        )
        
        # 2. POWER HEAD: Predicts local power contribution
        self.power_mlp = nn.Sequential(
            nn.Linear(feat_dim + 32, hidden//2),
            nn.ReLU(),
            nn.Linear(hidden//2, 1)
        )
        
        # 3. WL HEAD: Predicts local wirelength contribution
        self.wl_mlp = nn.Sequential(
            nn.Linear(feat_dim + 32, hidden//2), 
            nn.ReLU(),
            nn.Linear(hidden//2, 1)
        )

    def forward(self, X_tilde, cts_knobs):
        # A. Process Knobs: [1, num_knobs] -> [1, 32]
        k_feat = F.relu(self.knob_proj(cts_knobs)) 
        
        # B. Broadcast knobs to every supernode
        K = X_tilde.size(0)
        # Expand k_feat so every node gets the same tool settings: [K, 32]
        k_feat_expanded = k_feat.expand(K, -1) 
        
        # Glue the node features to the knobs: [K, feat_dim + 32]
        node_inputs = torch.cat([X_tilde, k_feat_expanded], dim=-1)
        
        # C. Local Predictions (Apply MLPs to EACH supernode independently)
        local_skew = self.skew_mlp(node_inputs)   # [K, 1]
        local_power = self.power_mlp(node_inputs) # [K, 1]
        local_wl = self.wl_mlp(node_inputs)       # [K, 1]
        
        # D. Global Aggregation (The "Bottom-Up" Assembly)
        # Skew is dictated by the worst-case local path (Max)
        global_skew = torch.max(local_skew, dim=0, keepdim=True)[0] # [1, 1]
        
        # Power is additive across the whole chip (Sum)
        global_power = torch.sum(local_power, dim=0, keepdim=True)  # [1, 1]
        
        # Wirelength is additive across the whole chip (Sum)
        global_wl = torch.sum(local_wl, dim=0, keepdim=True)        # [1, 1]
        
        return global_skew, global_power, global_wl

In [6]:
"""
Phase 2: Task-Aware CTS Predictor
===================================
Full implementation with:
- One-hop enrichment (timing + physical context)
- Edge-informed per-supernode scoring
- Physics-based WL estimation
- Design-level descriptors
- Soft-max aggregation for skew
- Three task heads with proper inductive bias
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import os


# ==============================================================================
# HELPER: Extract everything useful from the compressed graph
# ==============================================================================

def enrich_and_extract(X_tilde, A_tilde_skip, A_tilde_wire):
    """
    Takes the compressed graph and extracts:
    1. X_full  [k, 84]  — per-supernode features enriched by 1-hop neighbors
    2. edge_scores dict — per-supernode edge-based scores for each task
    3. design_desc [1, 15] — global circuit fingerprint
    """
    k = X_tilde.shape[0]
    device = X_tilde.device
    eps = 1e-8

    # ----------------------------------------------------------------
    # STEP 1: One-hop propagation
    # Each supernode learns what its timing and physical neighbors look like
    # ----------------------------------------------------------------
    skip_rowsum = A_tilde_skip.sum(dim=1, keepdim=True).clamp(min=eps)
    wire_rowsum = A_tilde_wire.sum(dim=1, keepdim=True).clamp(min=eps)

    skip_norm = A_tilde_skip / skip_rowsum   # [k, k] row-normalized
    wire_norm = A_tilde_wire / wire_rowsum   # [k, k] row-normalized

    X_timing   = skip_norm @ X_tilde         # [k, 28] each node sees timing neighbors
    X_physical = wire_norm @ X_tilde         # [k, 28] each node sees physical neighbors

    X_full = torch.cat([X_tilde, X_timing, X_physical], dim=-1)  # [k, 84]

    # ----------------------------------------------------------------
    # STEP 2: Edge-informed per-supernode scores
    # These tell each head HOW MUCH each supernode matters for its task
    # ----------------------------------------------------------------

    # -- SKEW: how timing-connected is each supernode --
    # A supernode that sits on many heavy timing paths = high skew contributor
    skip_no_diag = A_tilde_skip.clone()
    skip_no_diag.fill_diagonal_(0)
    skip_degree = skip_no_diag.sum(dim=1, keepdim=True)  # [k, 1]

    # FF density of this supernode (col 10 = is_sequential)
    ff_density = X_tilde[:, 10:11]  # [k, 1]

    # Distance to timing neighbors (physically far timing connections = more skew)
    coords = X_tilde[:, 0:2]  # [k, 2]
    src_c  = coords.unsqueeze(1).expand(-1, k, -1)   # [k, k, 2]
    dst_c  = coords.unsqueeze(0).expand(k, -1, -1)   # [k, k, 2]
    pairwise_dist = torch.norm(src_c - dst_c, dim=-1) # [k, k]

    # Weighted average distance to timing neighbors
    timing_reach = (skip_norm * pairwise_dist).sum(dim=1, keepdim=True)  # [k, 1]

    # Combined skew score per supernode: FF-dense + highly connected + far timing reach
    skew_node_score = ff_density * skip_degree * (1.0 + timing_reach)  # [k, 1]

    # -- POWER: how much each supernode drives --
    # wire_degree = how many physical connections = how much buffering = power
    wire_degree = A_tilde_wire.sum(dim=1, keepdim=True)  # [k, 1]
    activity    = X_tilde[:, 12:14].sum(dim=1, keepdim=True)  # [k, 1] toggle activity
    cap         = X_tilde[:, 7:9].sum(dim=1, keepdim=True)    # [k, 1] pin capacitance
    power_node_score = wire_degree * activity * cap             # [k, 1]

    # -- WL: local routing length estimate per supernode --
    # sum_j( A_wire[i,j] * ||coord_i - coord_j|| )
    # This is literally the estimated wire length touching supernode i
    wl_node_score = (A_tilde_wire * pairwise_dist).sum(dim=1, keepdim=True)  # [k, 1]

    edge_scores = {
        'skew':  skew_node_score,   # [k, 1]
        'power': power_node_score,  # [k, 1]
        'wl':    wl_node_score,     # [k, 1]
    }

    # ----------------------------------------------------------------
    # STEP 3: Design-level descriptors
    # Global fingerprint that distinguishes different circuits
    # even when supernode features happen to be similar
    # ----------------------------------------------------------------
    spread  = X_tilde[:, -1]   # geometric spread per cluster
    count   = X_tilde[:, -2]   # total cell area per cluster
    buf_den = X_tilde[:, 11]   # buffer density

    # Complexity
    compression_ratio     = torch.tensor(float(k), device=device) / (count.sum() + eps)
    avg_cluster_size      = count.mean()
    cluster_size_var      = count.var()
    total_ff_density      = ff_density.mean()
    total_buf_density     = buf_den.mean()

    # Connectivity
    wire_density          = (A_tilde_wire > 0).float().mean()
    skip_density          = (skip_no_diag > 0).float().mean()
    avg_wire_degree       = wire_degree.mean()
    avg_skip_degree       = skip_degree.mean()
    wire_degree_var       = wire_degree.var()
    skip_degree_var       = skip_degree.var()

    # Spatial
    coord_range_x         = coords[:, 0].max() - coords[:, 0].min()
    coord_range_y         = coords[:, 1].max() - coords[:, 1].min()
    centroid              = coords.mean(dim=0)
    avg_dist_to_centroid  = torch.norm(coords - centroid, dim=1).mean()

    # Timing complexity: skip edges per FF
    n_ff_approx           = ff_density.sum().clamp(min=eps)
    n_skip_edges          = (skip_no_diag > 0).float().sum()
    timing_complexity     = n_skip_edges / n_ff_approx

    design_desc = torch.stack([
        compression_ratio, avg_cluster_size, cluster_size_var,
        total_ff_density,  total_buf_density,
        wire_density,      skip_density,
        avg_wire_degree.squeeze(), avg_skip_degree.squeeze(),
        wire_degree_var.squeeze(), skip_degree_var.squeeze(),
        coord_range_x,     coord_range_y,
        avg_dist_to_centroid, timing_complexity
    ]).unsqueeze(0)  # [1, 15]

    # Replace any NaN/Inf that can appear from empty graphs
    design_desc = torch.nan_to_num(design_desc, nan=0.0, posinf=1.0, neginf=0.0)

    return X_full, edge_scores, design_desc


# ==============================================================================
# MAIN MODEL
# ==============================================================================

class TaskPredictor(nn.Module):
    """
    Three-head predictor for CTS outcomes (skew, power, wirelength).

    Architecture per head:
      1. Shared encoder: [k, 84] → [k, hidden]
      2. Edge-biased attention: score each supernode by task relevance
      3. Soft aggregation: weighted sum → [1, hidden]
      4. MLP: cat(pooled, design_desc, knobs) → scalar prediction

    Inductive biases:
      - Skew:  soft-max over timing-connected FF-dense supernodes
      - Power: weighted sum over high-activity high-degree supernodes
      - WL:    sum of local routing estimates + coordinate spread
    """

    def __init__(self, feat_dim=84, num_knobs=4, hidden=128, desc_dim=15):
        super().__init__()

        # Knob projection — same for all heads
        self.knob_proj = nn.Sequential(
            nn.Linear(num_knobs, 32),
            nn.ReLU()
        )

        # Design descriptor projection — same for all heads
        self.desc_proj = nn.Sequential(
            nn.Linear(desc_dim, 32),
            nn.ReLU()
        )

        # Shared encoder — all heads read from this
        self.encoder = nn.Sequential(
            nn.Linear(feat_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU()
        )

        # Task-specific attention query + edge bias weight
        # query: learned direction in hidden space → which supernodes matter
        # edge_bias_w: learned scalar → how much to trust edge score vs features
        self.skew_query      = nn.Linear(hidden, 1, bias=False)
        self.skew_edge_bias  = nn.Parameter(torch.tensor(0.1))

        self.power_query     = nn.Linear(hidden, 1, bias=False)
        self.power_edge_bias = nn.Parameter(torch.tensor(0.1))

        self.wl_query        = nn.Linear(hidden, 1, bias=False)
        self.wl_edge_bias    = nn.Parameter(torch.tensor(0.1))

        # Head input: hidden(128) + desc(32) + knobs(32) = 192
        head_in = hidden + 32 + 32

        self.skew_head = nn.Sequential(
            nn.Linear(head_in, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        self.power_head = nn.Sequential(
            nn.Linear(head_in, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        # WL head gets extra direct physics features
        # hidden + desc(32) + knobs(32) + wl_direct(3) = 195
        self.wl_head = nn.Sequential(
            nn.Linear(head_in + 3, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, X_tilde, A_tilde_skip, A_tilde_wire, cts_knobs):
        """
        X_tilde:      [k, 28]
        A_tilde_skip: [k, k]
        A_tilde_wire: [k, k]
        cts_knobs:    [1, 4]
        """
        eps = 1e-8
        coords = X_tilde[:, 0:2]

        # -- Extract enriched features and scores --
        X_full, edge_scores, design_desc = enrich_and_extract(
            X_tilde, A_tilde_skip, A_tilde_wire
        )

        # -- Project knobs and design descriptor --
        k_feat = self.knob_proj(cts_knobs)       # [1, 32]
        d_feat = self.desc_proj(design_desc)     # [1, 32]
        shared = torch.cat([k_feat, d_feat], dim=-1)  # [1, 64]

        # -- Encode all supernodes --
        h = self.encoder(X_full)  # [k, hidden]

        # ============================================================
        # SKEW HEAD
        # Soft-max aggregation biased toward timing-critical supernodes
        # ============================================================
        skew_feat_score = self.skew_query(h)               # [k, 1]
        skew_edge_score = edge_scores['skew']              # [k, 1]
        # Normalize edge score to same scale as feature score
        skew_edge_norm  = skew_edge_score / (skew_edge_score.max() + eps)
        skew_combined   = skew_feat_score + self.skew_edge_bias * skew_edge_norm
        skew_attn       = F.softmax(skew_combined, dim=0)  # [k, 1]
        h_skew          = (skew_attn * h).sum(dim=0, keepdim=True)  # [1, hidden]

        skew_pred = self.skew_head(
            torch.cat([h_skew, shared], dim=-1)
        )  # [1, 1]

        # ============================================================
        # POWER HEAD
        # Weighted sum biased toward high-activity, high-degree supernodes
        # ============================================================
        power_feat_score = self.power_query(h)              # [k, 1]
        power_edge_score = edge_scores['power']             # [k, 1]
        power_edge_norm  = power_edge_score / (power_edge_score.max() + eps)
        power_combined   = power_feat_score + self.power_edge_bias * power_edge_norm
        power_attn       = F.softmax(power_combined, dim=0) # [k, 1]
        h_power          = (power_attn * h).sum(dim=0, keepdim=True)  # [1, hidden]

        power_pred = self.power_head(
            torch.cat([h_power, shared], dim=-1)
        )  # [1, 1]

        # ============================================================
        # WL HEAD
        # Sum of local routing estimates + coordinate spread
        # Direct physics estimate fed alongside learned embedding
        # ============================================================
        wl_feat_score = self.wl_query(h)                   # [k, 1]
        wl_edge_score = edge_scores['wl']                  # [k, 1]
        wl_edge_norm  = wl_edge_score / (wl_edge_score.max() + eps)
        wl_combined   = wl_feat_score + self.wl_edge_bias * wl_edge_norm
        wl_attn       = F.softmax(wl_combined, dim=0)      # [k, 1]
        h_wl          = (wl_attn * h).sum(dim=0, keepdim=True)   # [1, hidden]

        # Direct physics features for WL
        # 1. Total estimated routing length in compressed graph
        total_routing = wl_edge_score.sum().unsqueeze(0).unsqueeze(0)  # [1, 1]
        # 2. Coordinate variance = physical spread of supernodes
        coord_var     = coords.var(dim=0).mean().unsqueeze(0).unsqueeze(0)  # [1, 1]
        # 3. FF-weighted spread
        ff_w          = X_tilde[:, 10] / (X_tilde[:, 10].sum() + eps)
        ff_spread     = (ff_w.unsqueeze(1) * coords).var(dim=0).mean()
        ff_spread     = ff_spread.unsqueeze(0).unsqueeze(0)  # [1, 1]

        wl_direct = torch.cat([total_routing, coord_var, ff_spread], dim=-1)  # [1, 3]
        wl_direct = torch.nan_to_num(wl_direct, nan=0.0)

        wl_pred = self.wl_head(
            torch.cat([h_wl, shared, wl_direct], dim=-1)
        )  # [1, 1]

        return skew_pred, power_pred, wl_pred


# ==============================================================================
# TRAINING LOOP
# ==============================================================================

def train_phase2(
    phase1_model,
    train_cache,
    test_cache,
    csv_path,
    device,
    num_epochs=150,
    lr=1e-3,
    save_path="phase2_predictor.pt"
):
    from helper import get_compressed_graph, load_cts_parameters, relative_masking

    predictor = TaskPredictor(feat_dim=84).to(device)
    optimizer = torch.optim.Adam(predictor.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )
    criterion = nn.MSELoss()

    best_val_mae = float('inf')

    print(f"\n🚀 Phase 2 Training | {len(train_cache)} train | {len(test_cache)} test\n")

    for epoch in range(num_epochs):
        # ----------------------------------------------------------------
        # TRAIN
        # ----------------------------------------------------------------
        predictor.train()
        t_stats = {'mse': 0, 'mae': 0,
                   'mae_s': 0, 'mae_p': 0, 'mae_w': 0, 'cnt': 0}
        random.shuffle(train_cache)

        for step, (filename, data) in enumerate(train_cache):
            placement_id = filename.replace('.pt', '')

            # Frozen phase 1 extraction
            with torch.no_grad():
                X          = data['X'].to(device)
                X_cell_ids = data['X_cell_ids'].to(device)
                p_indices  = data['p_indices'].to(device)
                A_skip_csr = data['A_skip_csr'].to(device)
                A_wire_csr = data['A_wire_csr'].to(device)
                raw_areas  = data['raw_areas'].to(device)

                _, _, _, _, C, X_combined = phase1_model(
                    X, X_cell_ids, data['num_nodes'],
                    p_indices, A_skip_csr, data['current_k'], tau=0.5
                )
                X_tilde, A_tilde_skip, A_tilde_wire = get_compressed_graph(
                    X_combined, C, A_skip_csr, A_wire_csr, raw_areas
                )
                # Clamp adjacencies to avoid numerical issues
                A_tilde_skip = A_tilde_skip.clamp(min=0)
                A_tilde_wire = A_tilde_wire.clamp(min=0)

            cts_configs = load_cts_parameters(csv_path, placement_id, device)
            if not cts_configs:
                continue

            for run in cts_configs:
                knobs   = run['knobs'].unsqueeze(0).to(device)   # [1, 4]
                targets = run['targets']

                s_pred, p_pred, w_pred = predictor(
                    X_tilde, A_tilde_skip, A_tilde_wire, knobs
                )

                preds = torch.cat([s_pred.view(-1),
                                   p_pred.view(-1),
                                   w_pred.view(-1)])
                trues = torch.cat([targets['skew'].view(-1).to(device),
                                   targets['power'].view(-1).to(device),
                                   targets['wl'].view(-1).to(device)])

                loss = criterion(preds, trues)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(predictor.parameters(), max_norm=1.0)
                optimizer.step()

                with torch.no_grad():
                    abs_err = torch.abs(preds - trues)
                    t_stats['mse']   += loss.item()
                    t_stats['mae']   += abs_err.mean().item()
                    t_stats['mae_s'] += abs_err[0].item()
                    t_stats['mae_p'] += abs_err[1].item()
                    t_stats['mae_w'] += abs_err[2].item()
                    t_stats['cnt']   += 1

            del X, X_cell_ids, p_indices, A_skip_csr, A_wire_csr
            del C, X_combined, X_tilde, A_tilde_skip, A_tilde_wire, raw_areas
            if step % 5 == 0:
                torch.cuda.empty_cache()

        # ----------------------------------------------------------------
        # VALIDATE
        # ----------------------------------------------------------------
        predictor.eval()
        v_stats = {'mse': 0, 'mae': 0,
                   'mae_s': 0, 'mae_p': 0, 'mae_w': 0, 'cnt': 0}

        with torch.no_grad():
            for filename, data in test_cache:
                placement_id = filename.replace('.pt', '')

                X          = data['X'].to(device)
                X_cell_ids = data['X_cell_ids'].to(device)
                p_indices  = data['p_indices'].to(device)
                A_skip_csr = data['A_skip_csr'].to(device)
                A_wire_csr = data['A_wire_csr'].to(device)
                raw_areas  = data['raw_areas'].to(device)

                _, _, _, _, C, X_combined = phase1_model(
                    X, X_cell_ids, data['num_nodes'],
                    p_indices, A_skip_csr, data['current_k'], tau=0.5
                )
                X_tilde, A_tilde_skip, A_tilde_wire = get_compressed_graph(
                    X_combined, C, A_skip_csr, A_wire_csr, raw_areas
                )
                A_tilde_skip = A_tilde_skip.clamp(min=0)
                A_tilde_wire = A_tilde_wire.clamp(min=0)

                cts_configs = load_cts_parameters(csv_path, placement_id, device)
                if not cts_configs:
                    continue

                for run in cts_configs:
                    knobs   = run['knobs'].unsqueeze(0).to(device)
                    targets = run['targets']

                    s_p, p_p, w_p = predictor(
                        X_tilde, A_tilde_skip, A_tilde_wire, knobs
                    )

                    val_preds = torch.cat([s_p.view(-1),
                                           p_p.view(-1),
                                           w_p.view(-1)])
                    val_trues = torch.cat([targets['skew'].view(-1).to(device),
                                           targets['power'].view(-1).to(device),
                                           targets['wl'].view(-1).to(device)])
                    v_loss = criterion(val_preds, val_trues)

                    abs_err = torch.abs(val_preds - val_trues)
                    v_stats['mse']   += v_loss.item()
                    v_stats['mae']   += abs_err.mean().item()
                    v_stats['mae_s'] += abs_err[0].item()
                    v_stats['mae_p'] += abs_err[1].item()
                    v_stats['mae_w'] += abs_err[2].item()
                    v_stats['cnt']   += 1

                del X, X_cell_ids, p_indices, A_skip_csr, A_wire_csr
                del C, X_combined, X_tilde, A_tilde_skip, A_tilde_wire, raw_areas

        def avg(s, k): return s[k] / s['cnt'] if s['cnt'] > 0 else 0.0

        val_mae = avg(v_stats, 'mae')
        scheduler.step(val_mae)

        # Save best model
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            torch.save(predictor.state_dict(), save_path)

        # ----------------------------------------------------------------
        # LOGGING
        # ----------------------------------------------------------------
        print(f"\n📊 [EPOCH {epoch+1:03d}/{num_epochs}]")
        print(f"{'Metric':<12} | {'Train':<12} | {'Val (unseen)':<12}")
        print("-" * 42)
        print(f"{'MSE':<12} | {avg(t_stats,'mse'):<12.6f} | {avg(v_stats,'mse'):<12.6f}")
        print(f"{'MAE':<12} | {avg(t_stats,'mae'):<12.5f} | {avg(v_stats,'mae'):<12.5f}")
        print(f"{'Skew MAE':<12} | {avg(t_stats,'mae_s'):<12.5f} | {avg(v_stats,'mae_s'):<12.5f}")
        print(f"{'Power MAE':<12} | {avg(t_stats,'mae_p'):<12.5f} | {avg(v_stats,'mae_p'):<12.5f}")
        print(f"{'WL MAE':<12} | {avg(t_stats,'mae_w'):<12.5f} | {avg(v_stats,'mae_w'):<12.5f}")
        print(f"  best val MAE so far: {best_val_mae:.5f}")
        print("=" * 42)

    print(f"\n✅ Training complete. Best val MAE: {best_val_mae:.5f}")
    print(f"   Model saved to: {save_path}")
    return predictor


# ==============================================================================
# QUICK SANITY CHECK — run this before training to verify signal exists
# ==============================================================================

def sanity_check(phase1_model, train_cache, csv_path, device):
    """
    Verifies that the physics features extracted from X_tilde
    actually correlate with the targets.
    If they don't correlate here, no architecture will fix it.
    """
    from helper import get_compressed_graph, load_cts_parameters

    print("\n🔍 SANITY CHECK: Physics feature vs target correlation")
    print("=" * 60)

    routing_estimates = []
    coord_vars        = []
    wl_targets        = []
    activity_sums     = []
    power_targets     = []
    timing_dists      = []
    skew_targets      = []

    with torch.no_grad():
        for filename, data in train_cache:
            placement_id = filename.replace('.pt', '')

            X          = data['X'].to(device)
            X_cell_ids = data['X_cell_ids'].to(device)
            p_indices  = data['p_indices'].to(device)
            A_skip_csr = data['A_skip_csr'].to(device)
            A_wire_csr = data['A_wire_csr'].to(device)
            raw_areas  = data['raw_areas'].to(device)

            _, _, _, _, C, X_combined = phase1_model(
                X, X_cell_ids, data['num_nodes'],
                p_indices, A_skip_csr, data['current_k'], tau=0.5
            )
            X_tilde, A_tilde_skip, A_tilde_wire = get_compressed_graph(
                X_combined, C, A_skip_csr, A_wire_csr, raw_areas
            )
            A_tilde_wire  = A_tilde_wire.clamp(min=0)
            A_tilde_skip  = A_tilde_skip.clamp(min=0)

            k      = X_tilde.shape[0]
            coords = X_tilde[:, 0:2]

            # Physics features
            src_c = coords.unsqueeze(1).expand(-1, k, -1)
            dst_c = coords.unsqueeze(0).expand(k, -1, -1)
            dist  = torch.norm(src_c - dst_c, dim=-1)

            skip_no_diag = A_tilde_skip.clone()
            skip_no_diag.fill_diagonal_(0)

            routing_est  = (A_tilde_wire * dist).sum().item()
            coord_var    = coords.var(dim=0).mean().item()
            activity_sum = X_tilde[:, 12:14].sum().item()

            # Timing: worst skip edge distance
            if skip_no_diag.max() > 0:
                flat      = skip_no_diag.view(-1)
                top_idx   = flat.argmax()
                r, c      = top_idx // k, top_idx % k
                timing_d  = dist[r, c].item()
            else:
                timing_d  = 0.0

            cts_configs = load_cts_parameters(csv_path, placement_id, device)
            if not cts_configs:
                continue

            # Use first config as representative for this design
            run = cts_configs[0]
            routing_estimates.append(routing_est)
            coord_vars.append(coord_var)
            wl_targets.append(run['targets']['wl'].item())
            activity_sums.append(activity_sum)
            power_targets.append(run['targets']['power'].item())
            timing_dists.append(timing_d)
            skew_targets.append(run['targets']['skew'].item())

            del X, X_cell_ids, p_indices, A_skip_csr, A_wire_csr
            del C, X_combined, X_tilde, A_tilde_skip, A_tilde_wire, raw_areas

    def pearson(x, y):
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        xm = x - x.mean()
        ym = y - y.mean()
        return (xm * ym).sum() / (xm.norm() * ym.norm() + 1e-8)

    r_wl_route  = pearson(routing_estimates, wl_targets)
    r_wl_var    = pearson(coord_vars, wl_targets)
    r_pow_act   = pearson(activity_sums, power_targets)
    r_skew_dist = pearson(timing_dists, skew_targets)

    print(f"WL    ← routing_estimate  : r = {r_wl_route:.3f}  {'✅' if abs(r_wl_route) > 0.4 else '⚠️'}")
    print(f"WL    ← coord_variance    : r = {r_wl_var:.3f}    {'✅' if abs(r_wl_var) > 0.4 else '⚠️'}")
    print(f"Power ← activity_sum      : r = {r_pow_act:.3f}   {'✅' if abs(r_pow_act) > 0.4 else '⚠️'}")
    print(f"Skew  ← timing_path_dist  : r = {r_skew_dist:.3f} {'✅' if abs(r_skew_dist) > 0.4 else '⚠️'}")
    print("\nIf r < 0.4 for any row, the compressed graph is not preserving")
    print("that physical property. Improve phase 1 before continuing.")
    print("=" * 60)


# ==============================================================================
# ENTRY POINT
# ==============================================================================

if __name__ == "__main__":
    # ---- Setup ----
    import torch.optim as optim
    device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    csv_path = "./dataset_with_def/unified_manifest_normalized.csv"

    # Load metadata and phase 1 model
    metadata           = torch.load("global_metadata.pt")
    global_max_k       = metadata['global_max_k']
    global_max_cell_types = metadata['global_max_cell_types']

    # Import your existing classes
          # your phase 1 model
    from helper import get_compressed_graph, load_cts_parameters

    phase1_model = FirstTerm(
        num_cell_types=global_max_cell_types,
        num_of_clusters=global_max_k
    ).to(device)
    phase1_model.load_state_dict(torch.load("phase1_clustering_ep250_twohop.pt"))
    phase1_model.eval()
    for p in phase1_model.parameters():
        p.requires_grad = False

    # ---- Build cache ----
    processed_dir = "processed_graphs"
    all_files     = [f for f in os.listdir(processed_dir) if f.endswith('.pt')]

    # Group by design family for balanced split
    grouped = {}
    for f in all_files:
        name = f.split('_run_')[0]
        grouped.setdefault(name, []).append(f)

    train_files = []
    test_files  = []
    for name, files in grouped.items():
        random.shuffle(files)
        # Hold out last 2 per family for test
        test_files.extend(files[-2:])
        train_files.extend(files[:-2])

    print(f"Train: {len(train_files)} | Test: {len(test_files)}")

    def build_cache(file_list):
        cache = []
        for f in file_list:
            data = torch.load(os.path.join(processed_dir, f))
            cache.append((f, data))
        return cache

    train_cache = build_cache(train_files)
    test_cache  = build_cache(test_files)

    # ---- Sanity check first ----
    sanity_check(phase1_model, train_cache, csv_path, device)

    # ---- Train ----
    predictor = train_phase2(
        phase1_model  = phase1_model,
        train_cache   = train_cache,
        test_cache    = test_cache,
        csv_path      = csv_path,
        device        = device,
        num_epochs    = 150,
        lr            = 1e-3,
        save_path     = "phase2_predictor.pt"
    )

Train: 132 | Test: 8

🔍 SANITY CHECK: Physics feature vs target correlation
WL    ← routing_estimate  : r = 0.947  ✅
WL    ← coord_variance    : r = 0.055    ⚠️
Power ← activity_sum      : r = 0.202   ⚠️
Skew  ← timing_path_dist  : r = 0.007 ⚠️

If r < 0.4 for any row, the compressed graph is not preserving
that physical property. Improve phase 1 before continuing.

🚀 Phase 2 Training | 132 train | 8 test


📊 [EPOCH 001/150]
Metric       | Train        | Val (unseen)
------------------------------------------
MSE          | 1228780612899.522705 | 2.262628    
MAE          | 72340.82063  | 1.02796     
Skew MAE     | 22378.53970  | 0.68238     
Power MAE    | 43297.77861  | 0.87031     
WL MAE       | 151346.14283 | 1.53120     
  best val MAE so far: 1.02796

📊 [EPOCH 002/150]
Metric       | Train        | Val (unseen)
------------------------------------------
MSE          | 1465.167283  | 0.952582    
MAE          | 2.34744      | 0.82474     
Skew MAE     | 1.01915      | 0.68119   

KeyboardInterrupt: 

In [ ]:
# ==========================================
# 3. Final Evaluation & True vs. Predicted
# ==========================================
print("\n" + "="*50)
print("🚀 FINAL EVALUATION: TRUE VS PREDICTED")
print("="*50)

predictor_model.eval()

mae_skew_sum, mae_power_sum, mae_wl_sum = 0, 0, 0
eval_preds = 0
print_limit = 15 # Only print the first 15 samples to keep the console clean

with torch.no_grad():
    for filename, data in ram_cache:
        placement_id = filename.replace('.pt', '')
        cts_runs = load_cts_parameters(csv_file, placement_id, device)
        if not cts_runs: 
            continue

        # 1. Get Clustering Matrix (Frozen)
        _, _, _, _, C, X_combined = phase1_model(
            data['X'].to(device), data['X_cell_ids'].to(device), 
            data['num_nodes'], data['p_indices'].to(device), 
            data['A_skip_csr'].to(device), data['current_k'], tau=0.85
        )

        # 2. Evaluate CTS Runs
        for run in cts_runs:
            cts_params = run['knobs'].view(1, -1)
            t_p = run['targets']['power'].view(1, 1)
            t_s = run['targets']['skew'].view(1, 1)
            t_w = run['targets']['wl'].view(1, 1)
            
            # Predict
            skew_pred, power_pred, wl_pred = predictor_model(C, X_combined, cts_params)
            
            # Calculate Absolute Errors
            mae_skew_sum += abs(skew_pred.item() - t_s.item())
            mae_power_sum += abs(power_pred.item() - t_p.item())
            mae_wl_sum += abs(wl_pred.item() - t_w.item())
            
            # Print sample True vs Predicted
            if eval_preds < print_limit:
                print(f"Sample {eval_preds+1:2} | "
                      f"SKEW [True: {t_s.item():>6.3f} | Pred: {skew_pred.item():>6.3f}]  ---  "
                      f"POWER [True: {t_p.item():>6.3f} | Pred: {power_pred.item():>6.3f}]")
                
            eval_preds += 1
            
        del C, X_combined
        torch.cuda.empty_cache()

# 3. Calculate and Print Global MAE
final_mae_skew = mae_skew_sum / max(eval_preds, 1)
final_mae_power = mae_power_sum / max(eval_preds, 1)
final_mae_wl = mae_wl_sum / max(eval_preds, 1)

print("-" * 50)
print(f"🎯 FINAL GLOBAL MAE (Across {eval_preds} CTS Runs)")
print(f"   ➤ Skew MAE:  {final_mae_skew:.4f}")
print(f"   ➤ Power MAE: {final_mae_power:.4f}")
print(f"   ➤ WL MAE:    {final_mae_wl:.4f}")
print("="*50)

# Save the predictor once it's trained and evaluated
torch.save(predictor_model.state_dict(), "phase2_predictor.pt")
print("✅ Saved phase2_predictor.pt")


🚀 FINAL EVALUATION: TRUE VS PREDICTED
Sample  1 | SKEW [True: -0.245 | Pred: -0.496]  ---  POWER [True: -1.322 | Pred: -1.433]
Sample  2 | SKEW [True:  1.612 | Pred:  1.427]  ---  POWER [True: -1.237 | Pred: -1.334]
Sample  3 | SKEW [True:  1.392 | Pred:  1.356]  ---  POWER [True: -1.335 | Pred: -1.375]
Sample  4 | SKEW [True: -0.320 | Pred: -0.287]  ---  POWER [True: -1.159 | Pred: -1.120]
Sample  5 | SKEW [True: -0.158 | Pred: -0.272]  ---  POWER [True: -1.236 | Pred: -1.282]
Sample  6 | SKEW [True:  0.981 | Pred:  1.120]  ---  POWER [True: -1.375 | Pred: -1.430]
Sample  7 | SKEW [True: -0.440 | Pred: -0.412]  ---  POWER [True: -1.244 | Pred: -1.289]
Sample  8 | SKEW [True: -0.221 | Pred: -0.418]  ---  POWER [True: -1.239 | Pred: -1.164]
Sample  9 | SKEW [True:  2.027 | Pred:  2.220]  ---  POWER [True: -1.331 | Pred: -1.388]
Sample 10 | SKEW [True: -1.149 | Pred: -0.995]  ---  POWER [True: -1.356 | Pred: -1.324]
Sample 11 | SKEW [True: -0.198 | Pred: -0.368]  ---  POWER [True:  0.78

Train: 132 | Test: 8

🔍 SANITY CHECK: Physics feature vs target correlation
WL    ← routing_estimate  : r = 0.954  ✅
WL    ← coord_variance    : r = 0.045    ⚠️
Power ← activity_sum      : r = 0.207   ⚠️
Skew  ← timing_path_dist  : r = 0.011 ⚠️

If r < 0.4 for any row, the compressed graph is not preserving
that physical property. Improve phase 1 before continuing.


TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'